# GSM8K Rollout Lab — Gemma 3 1B-IT

Browse all 1319 GSM8K test problems with their **32 packaged rollouts** from `google/gemma-3-1b-it`, **edit a problem** (with a new gold answer), and **re-run rollouts through the exact OLMES pipeline** that produced the originals (same task config, prompt template, chat formatting, sampling settings, and scoring).

**Requirements**
- A GPU runtime (`Runtime → Change runtime type → GPU`). Any Colab GPU works — the model is 1B params. On T4/V100 the model runs in fp16 instead of bf16 (no bf16 support on those GPUs).
- A Hugging Face token with access to [google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it) (accept the Gemma license on the model page), stored as a Colab secret named `HF_TOKEN` (key icon in the left sidebar).

**Notes on replication**: new rollouts use the identical OLMES code path and settings as the packaged ones, but sampling at temperature 0.6 is stochastic — expect the same *distribution*, not identical strings. An optional seed makes your own runs repeatable (the original runs were unseeded).


## 1. Setup (~5–10 min on a fresh VM)

Clones the repo and installs the vendored OLMES harness with vLLM. If imports fail afterwards, do `Runtime → Restart session` and re-run from step 2.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/prudhvirajn/gsm8k-rollout-lab"
REPO_DIR = Path("/content/gsm8k-rollout-lab")

if not REPO_DIR.exists():
    !git clone --depth 1 {REPO_URL} {str(REPO_DIR)}
%cd {str(REPO_DIR)}
!pip install -q -e "./olmes[gpu]"


## 2. Hugging Face authentication


In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["TOKENIZERS_PARALLELISM"] = "false"


## 3. Load the packaged rollouts and browse

Filter by passrate (e.g. set the range to 0–0.3 to look at hard problems) or search the question text, then step through the 32 rollouts of the selected problem.


In [ ]:
import sys
sys.path.insert(0, "/content/gsm8k-rollout-lab")

from gsm8k_rollout_lab import load_bundle, ProblemBrowser

problems = load_bundle()
print(f"{len(problems)} problems, {sum(p['n_rollouts'] for p in problems)} rollouts")

browser = ProblemBrowser(problems)
browser.display()


## 4. Load the model (once, ~2 min)

The vLLM model stays resident, so each edit-and-rerun below only pays generation time (~1–2 min for 32 rollouts).


In [ ]:
import torch
from gsm8k_rollout_lab import GSM8KRolloutRunner

overrides = {}
if torch.cuda.get_device_capability()[0] < 8:
    overrides["dtype"] = "float16"  # T4/V100 have no bf16 support

runner = GSM8KRolloutRunner(model_overrides=overrides)


## 5. Edit a problem and run rollouts

`Load selected problem` pulls the problem chosen in the browser above into the form. Edit the question, set the **new gold answer** (just the final number), pick generation settings, and hit `Run rollouts`. The result view shows the new passrate next to the original problem's, and you can step through the new completions.

Leave `seed` blank to match the original (unseeded) setup; set it to make a run repeatable (seeded runs generate one rollout per seed, which is slower).


In [ ]:
from gsm8k_rollout_lab import EditRunPanel

panel = EditRunPanel(runner, browser)
panel.display()


### Programmatic API (optional)

The same runner can be called directly — useful for sweeping several edits:


In [ ]:
# result = runner.run(
#     question="Janet's ducks lay 20 eggs per day. ...",
#     answer="26",            # bare final answer, or full CoT ending in '#### 26'
#     n_rollouts=32,
#     temperature=0.6,
#     top_p=0.95,
#     seed=None,
# )
# print(result["n_correct"], "/", result["n_rollouts"])


## 6. Save your runs

Every run is written to `rollout_runs/<timestamp>_<name>/` in native OLMES format (`task-000-gsm8k-predictions.jsonl`, `metrics.json`, the task spec, and a scored `result.json`), plus a compact `rollout_runs/history.jsonl`. Colab VMs are ephemeral — copy them to Drive before closing:


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r rollout_runs /content/drive/MyDrive/gsm8k_rollout_lab_runs
